In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

from bayesian_torch.layers.flipout_layers import conv_flipout as bnn_conv, linear_flipout as bnn_linear

class LinearMomentProp(nn.Module):
    def __init__(self, original_layer):
        super().__init__()

        self.in_features = original_layer.in_features
        self.out_features = original_layer.out_features

        self.w_mu = nn.Parameter(original_layer.mu_weight.clone()) 
        self.w_var = nn.Parameter(F.softplus(original_layer.rho_weight).pow(2))

        self.b_mu = nn.Parameter(original_layer.mu_bias.clone())
        self.b_var = nn.Parameter(F.softplus(original_layer.rho_bias).pow(2))

    def forward(self, x):
        x_mu = x[0,:]
        x_var = x[1,:]

        r_mu = F.linear(x_mu, self.w_mu, self.b_mu)

        r_var = F.linear(x_var, self.w_var, self.b_var) 
        r_var += F.linear(x_var, self.w_mu.pow(2)) 
        r_var += F.linear(x_mu.pow(2), self.w_var)

        return torch.stack((r_mu, r_var))


class Conv2dMomentProp(nn.Module):
    def __init__(self, original_layer):
        super().__init__()

        self.groups = original_layer.groups
        self.padding = original_layer.padding
        self.stride = original_layer.stride

        self.w_mu = nn.Parameter(original_layer.mu_kernel.clone())
        self.w_var = nn.Parameter(F.softplus(original_layer.rho_kernel).pow(2))

        if original_layer.mu_bias != None:
            self.b_mu = nn.Parameter(original_layer.mu_bias.clone())
            self.b_var = nn.Parameter(F.softplus(original_layer.rho_bias).pow(2))
        else:
            self.b_mu = None
            self.b_var = None

    def batch_norm_fold(self, bn_layer):
        bn_coef = bn_layer.weight / torch.sqrt(bn_layer.running_var + bn_layer.eps)
        bn_coef_k = bn_coef.view(-1,1,1,1).expand(self.w_mu.shape)

        self.w_mu = nn.Parameter(self.w_mu * bn_coef_k)
        self.w_var = nn.Parameter((torch.sqrt(self.w_var) * bn_coef_k).pow(2))
    
        if self.b_mu != None:
            self.b_mu = nn.Parameter((self.b_mu - bn_layer.running_mean) * bn_coef + bn_layer.bias)
            self.b_var = nn.Parameter((torch.sqrt(self.b_var) * bn_coef).pow(2))
        else:
            self.b_mu = nn.Parameter((0 - bn_layer.running_mean) * bn_coef + bn_layer.bias)
            self.b_var = nn.Parameter((0 * bn_coef).pow(2))

    def forward(self, x):
        x_mu = x[0,:]
        x_var = x[1,:]

        r_mu = F.conv2d(x_mu, self.w_mu, self.b_mu, stride=self.stride, padding=self.padding, groups=self.groups)
       
        r_var = F.conv2d(x_var, self.w_var, self.b_var, stride=self.stride, padding=self.padding, groups=self.groups) 
        r_var += F.conv2d(x_var, self.w_mu.pow(2), stride=self.stride, padding=self.padding, groups=self.groups)
        r_var += F.conv2d(x_mu.pow(2), self.w_var, stride=self.stride, padding=self.padding, groups=self.groups)

        return torch.stack((r_mu, r_var))

class BatchNorm2dMomentProp(nn.Module):
    def __init__(self, original_layer):
        super().__init__()
        self.gamma = nn.Parameter(original_layer.weight)
        self.beta = nn.Parameter(original_layer.bias)
        self.running_mean = nn.Buffer(original_layer.running_mean)
        self.running_var = nn.Buffer(original_layer.running_var)
        self.eps = original_layer.eps
        self.momentum = original_layer.momentum

    def forward(self, x):
        x_mu = x[0,:]
        x_var = x[1,:]

        # Defined p(z) = mean(p(x)) across batch dimension
        
        if self.training:
            batch_size = x_mu.shape[0]
            z_mu = x_mu.mean(dim=0)
            z_var = (x_mu.pow(2) + x_var).mean(dim=0) - x_mu.sum(0).pow(2) / (batch_size**2)
            
            # Update running mean and running var using momentum
            self.running_mean = (1 - self.momentum) * self.running_mean + self.momentum * z_mu
            self.running_var = (1 - self.momentum) * self.running_var + self.running_var * z_var
        else:
            z_mu = self.running_mean
            z_var = self.running_var

        r_mu = self.gamma * (x_mu - z_mu) / torch.sqrt(z_var + self.eps) + self.beta
        r_var = self.gamma.pow(2) * x_var / (z_var + self.eps)

        return torch.stack((r_mu, r_var))


class AvgPool2dMomentProp(nn.Module):
    def __init__(self, original_layer):
        super().__init__()
        self.kernel_size = original_layer.kernel_size
        self.stride = original_layer.stride
    
    def forward(self, x):
        x_mu = x[0,:]
        x_var = x[1,:]

        r_mu = F.avg_pool2d(x_mu, self.kernel_size, self.stride)
        r_var = F.avg_pool2d(x_var, self.kernel_size, self.stride) / (self.kernel_size**2)

        return torch.stack((r_mu, r_var))
    

class AdaptiveAvgPool2dMomentProp(nn.Module):
    def __init__(self, original_layer):
        super().__init__()
        self.output_size = original_layer.output_size
    
    def forward(self, x):
        x_mu = x[0,:]
        x_var = x[1,:]

        _, _, h, w = x_mu.shape
        ishape = torch.tensor([h, w])

        stride = torch.floor(ishape / self.output_size)
        kernel_size = int((ishape - (self.output_size - 1) * stride).prod())

        r_mu = F.adaptive_avg_pool2d(x_mu, self.output_size)
        r_var = F.adaptive_avg_pool2d(x_var, self.output_size) / (kernel_size**2)

        return torch.stack((r_mu, r_var))

def _online_mean_var(sample_f, nsamples):
    # Online shifted variance calculation (r, r2, k)
    k = sample_f()
    r = torch.zeros_like(k)
    r2 = torch.zeros_like(k)
    
    for _ in range(nsamples - 1):
        x_sample = sample_f()
        r += x_sample - k
        r2 += (x_sample - k).pow(2)

    r_mu = k + r / nsamples
    r_var = (r2 - r.pow(2) / nsamples) / (nsamples - 1)

    return torch.stack((r_mu, r_var))


class MaxPoolMomentProp(nn.Module):
    def __init__(self, original_layer, num_mc=100, use_simple=False, use_large=False):
        super().__init__()
        self.num_mc = num_mc
        self.use_simple = use_simple
        self.use_large = use_large
        self.kernel_size = original_layer.kernel_size
        self.stride = original_layer.stride
    
    def mc_forward(self, x):
        x_mu = x[0,:]
        x_std = torch.sqrt(x[1,:])

        def _large_f():
            s = torch.empty((self.num_mc, *x_mu[0,:].shape), device=x_mu.device)
            s = s.normal_().unsqueeze(1).expand(self.num_mc, *x_mu.shape)
            s = torch.addcmul(x_mu.unsqueeze(0).expand(self.num_mc, *x_mu.shape), s, x_std.unsqueeze(0).expand(self.num_mc, *x_std.shape))
            
            # Flat the samples and the batch in the same first dimension
            s = s.view(self.num_mc * s.shape[1], *s[0,0,:].shape)
            s = F.max_pool2d(s, self.kernel_size, self.stride)
            # Divide the first 2 dimensions again in samples, batch
            s = s.view(self.num_mc, x_mu.shape[0], *s[0].shape)

            return torch.stack((s.mean(dim=0), s.var(dim=0)))

        # Allocate tensor in device and share samples across batch
        def _batch_f():
            n = torch.empty_like(x_mu[0,:]).normal_().unsqueeze(0).expand(*x_mu.shape)
            s = n * x_std + x_mu
            return F.max_pool2d(s, self.kernel_size, self.stride)
    
        if self.use_large:
            return _large_f()
        else:
            return _online_mean_var(_batch_f, self.num_mc)
    
    def simple_forward(self, x):
        x_mu = x[0,:]
        x_var = x[1,:]

        r_mu, idx = F.max_pool2d(x_mu, self.kernel_size, self.stride, return_indices=True)
        mask = F.max_unpool2d(torch.ones_like(r_mu), idx, self.kernel_size, self.stride)
        r_var = F.max_pool2d(mask * x_var, self.kernel_size, self.stride)

        return torch.stack((r_mu, r_var))

    def forward(self, x):
        if self.use_simple:
            return self.simple_forward(x)
        else:
            return self.mc_forward(x)


class ReLUMomentProp(nn.Module):
    def __init__(self, num_mc=100, use_simple=False, use_large=False):
        super().__init__()
        self.num_mc = num_mc
        self.use_simple = use_simple
        self.use_large = use_large

    def mc_forward(self, x):
        x_mu = x[0,:]
        x_std = torch.sqrt(x[1,:])

        def _large_f():
            r = torch.empty((self.num_mc, *x_mu[0,:].shape), device=x_mu.device)
            r = r.normal_().unsqueeze(1).expand(self.num_mc, *x_mu.shape)
            r = torch.addcmul(x_mu.unsqueeze(0).expand(self.num_mc, *x_mu.shape), r, x_std.unsqueeze(0).expand(self.num_mc, *x_std.shape))
            r = F.relu(r)
            return torch.stack((r.mean(dim=0), r.var(dim=0)))

        def _batch_f():
            n = torch.empty_like(x_mu[0,:]).normal_().unsqueeze(0).expand(*x_mu.shape)
            s = n * x_std + x_mu
            return F.relu(s)
        
        if self.use_large:
            return _large_f()
        else:
            return _online_mean_var(_batch_f, self.num_mc)
    
    def simple_forward(self, x):
        x_mu = x[0,:]
        x_var = x[1,:]

        r_mu = F.relu(x_mu)
        r_var = x_var * (x_mu > 0)

        return torch.stack((r_mu, r_var))

    def forward(self, x):
        if self.use_simple:
            return self.simple_forward(x)
        else:
            return self.mc_forward(x)


class SoftmaxMomentProp(nn.Module):
    def __init__(self, num_mc=100):
        super().__init__()
        self.num_mc = num_mc

    def forward(self, x):
        x_mu = x[0,:]
        x_std = torch.sqrt(x[1,:])

        r = torch.empty((self.num_mc, *x_mu[0,:].shape), device=x_mu.device)
        r = r.normal_().unsqueeze(1).expand(self.num_mc, *x_mu.shape)
        r = torch.addcmul(x_mu.unsqueeze(0).expand(self.num_mc, *x_mu.shape), r, x_std.unsqueeze(0).expand(self.num_mc, *x_std.shape))
        return F.softmax(r, dim=2)


class BnnSampler(nn.Module):
    def __init__(self, layer, num_mc=100, return_mp=False, use_large=False):
        super().__init__()
        self.num_mc = num_mc 
        self.l = layer
        self.use_large = use_large
        self.return_mp = return_mp

    def forward(self, x):
        x_mu = x[0,:]
        x_std = torch.sqrt(x[1,:])

        r = []
        for _ in range(self.num_mc):
            s = torch.addcmul(x_mu, torch.empty_like(x_mu).normal_(), x_std)
            r.append(self.l(s))
        r = torch.stack(r)
        
        if self.return_mp:
            return torch.stack((r.mean(dim=0), r.var(dim=0)))
        else:
            return r

def bnn_to_mp(m, num_mc, use_simple=False, use_large=False):
    prev = None
    for name, t in list(m._modules.items()):
        # Recursive call (modules inside module)
        if m._modules[name]._modules:
            if isinstance(t, BnnSampler):
                pass
            else:
                bnn_to_mp(m._modules[name], num_mc=num_mc, use_simple=use_simple, use_large=use_large)

        if isinstance(t, bnn_linear.LinearFlipout):
            setattr(m, name, LinearMomentProp(t))
        elif isinstance(t, bnn_conv.Conv2dFlipout):
            setattr(m, name, Conv2dMomentProp(t))
        elif isinstance(t, nn.AvgPool2d):
            setattr(m, name, AvgPool2dMomentProp(t))
        elif isinstance(t, nn.AdaptiveAvgPool2d):
            setattr(m, name, AdaptiveAvgPool2dMomentProp(t))
        
        elif isinstance(t, nn.Sigmoid):
            setattr(m, name, BnnSampler(t, num_mc=num_mc, return_mp=True))
        elif isinstance(t, nn.SiLU):
            setattr(m, name, BnnSampler(t, num_mc=num_mc, return_mp=True))

        elif isinstance(t, nn.ReLU):
            l = ReLUMomentProp()
            l.use_large = use_large
            l.use_simple = use_simple
            l.num_mc = num_mc
            setattr(m, name, l)
        elif isinstance(t, nn.MaxPool2d):
            l =  MaxPoolMomentProp(t)
            l.use_large = use_large
            l.use_simple = use_simple
            l.num_mc = num_mc
            setattr(m, name, l)
        elif isinstance(t, nn.Softmax):
            l = SoftmaxMomentProp()
            l.num_mc = num_mc
            setattr(m, name, l)

        elif isinstance(t, nn.BatchNorm2d):
            prev_m, prev_name = prev
            prev_m._modules[prev_name].batch_norm_fold(t)
            setattr(m, name, nn.Identity())

        prev = (m, name)

def split_model_graph(model: nn.Module, n: int):
    """
    AI generated function.
    Splits a PyTorch model into two independently runnable models.
    Part B will contain the last N executed module calls.
    Part A will contain everything before that.
    """
    traced = torch.fx.symbolic_trace(model)
    nodes = list(traced.graph.nodes)
    
    # Find all actual module calls to determine the split point
    module_nodes = [node for node in nodes if node.op == 'call_module']
    if n >= len(module_nodes):
        raise ValueError(f"Cannot split off {n} layers; model only has {len(module_nodes)} module calls.")
        
    # The first node of Part B is the n-th module from the end
    first_node_of_b = module_nodes[-n]
    cut_idx = nodes.index(first_node_of_b)
    
    nodes_A = nodes[:cut_idx]
    nodes_B = nodes[cut_idx:]
    
    # 1. Identify "Boundary Nodes" 
    # (Tensors calculated in A that are needed in B)
    boundary_nodes = []
    for node in nodes_B:
        for in_node in node.all_input_nodes:
            if in_node in nodes_A and in_node not in boundary_nodes:
                boundary_nodes.append(in_node)
                
    # 2. Build Part A
    graph_A = torch.fx.Graph()
    env_A = {} # Maps old graph nodes to new Graph A nodes
    
    for node in nodes_A:
        env_A[node] = graph_A.node_copy(node, lambda x: env_A[x])
        
    # Set the outputs for Part A based on the boundary nodes we found
    output_args_A = tuple(env_A[node] for node in boundary_nodes)
    if len(output_args_A) == 1:
        graph_A.output(output_args_A[0])
    else:
        graph_A.output(output_args_A)
        
    model_A = torch.fx.GraphModule(traced, graph_A)
    
    # 3. Build Part B
    graph_B = torch.fx.Graph()
    env_B = {} # Maps old graph nodes to new Graph B nodes
    
    # Create input placeholders in B for the incoming tensors from A
    for node in boundary_nodes:
        env_B[node] = graph_B.placeholder(node.name)
        
    # Copy the remaining nodes into B
    for node in nodes_B:
        env_B[node] = graph_B.node_copy(node, lambda x: env_B[x])
        
    model_B = torch.fx.GraphModule(traced, graph_B)
    
    return model_A, model_B

def copy_model_params_by_name(src: nn.Module, dst: nn.Module):
    with torch.no_grad():
        
        dst_dict = dict(dst.named_parameters())
        src_dict = dict(src.named_parameters())
        
        for k in dst_dict:
            dst.get_parameter(k).copy_(src_dict[k])

        dst_dict = dict(dst.named_buffers())
        src_dict = dict(src.named_buffers())

        for k in dst_dict:
            dst.get_buffer(k).copy_(src_dict[k])


from torchvision.models import efficientnet_b0
from bayesian_torch.models.dnn_to_bnn import dnn_to_bnn

# BNN conversion hyperparmeters
bnn_prior_parameters = {
    "prior_mu": 0.0,
    "prior_sigma": 1.0,
    "posterior_mu_init": 0.0,
    "posterior_rho_init": -3.0,
    "type": "Flipout",  # Flipout or Reparameterization
    "moped_enable": False,  # True to initialize mu/sigma from the pretrained dnn weights
    "moped_delta": 0.5,
}

class MomentPropEfficientnet_b0(nn.Module):
    def __init__(self, original, num_mc=20):
        super().__init__()
        self.num_mc = num_mc

        new_model = efficientnet_b0()
        new_model.classifier[1] = nn.Linear(1280, 9)

        new_model.eval()
        head, tail = split_model_graph(new_model, 3)
        
        dnn_to_bnn(head, bnn_prior_parameters)
        self.head = head
        copy_model_params_by_name(original, self.head)
        bnn_to_mp(self.head, num_mc, False, False)

        dnn_to_bnn(tail, bnn_prior_parameters)
        copy_model_params_by_name(original, tail)
        self.tail = BnnSampler(tail, num_mc=num_mc)

    def forward(self, x: torch.Tensor):
        y = torch.stack((x, torch.zeros_like(x)))
        y = self.head(y)
        return self.tail(y)

In [ ]:
import json
import shutil
import time
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter
from openpyxl.drawing.image import Image as XLImage
import io
from bayesian_torch.models.dnn_to_bnn import dnn_to_bnn
from medmnist import INFO, PathMNIST
from torchvision.transforms import v2

# Config

CHECKPOINT   = "BayNoMoped_run0.pth"
REPEATS_LIST = [10, 20, 100]
BATCH_SIZE   = 128
NUM_CLASSES  = 9
OUTPUT_DIR   = Path("mi_runs_cache")
DEVICE       = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {DEVICE}")

OUTPUT_DIR.mkdir(exist_ok=True)

BNN_PARAMS = {
    "prior_mu": 0.0, "prior_sigma": 1.0,
    "posterior_mu_init": 0.0, "posterior_rho_init": -3.0,
    "type": "Reparameterization",
    "moped_enable": False, "moped_delta": 0.5,
}

# Dataset
transform = v2.Compose([
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])

loaders = {}
for split in ["train", "val", "test"]:
    ds = PathMNIST(split=split, download=True, size=28, transform=transform)
    loaders[split] = torch.utils.data.DataLoader(
        ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True
    )
class_names = INFO["pathmnist"]["label"]

# Models
def load_mc_model():
    net = torchvision.models.efficientnet_b0(progress=False)
    net.classifier[1] = nn.Linear(1280, 9)
    ckpt = torch.load(CHECKPOINT, map_location=DEVICE)
    dnn_to_bnn(net, BNN_PARAMS)
    net.load_state_dict(ckpt["model_state_dict"])
    net.to(DEVICE).eval()
    return net

def load_mp_model():
    net = torchvision.models.efficientnet_b0(progress=False)
    net.classifier[1] = nn.Linear(1280, 9)
    ckpt = torch.load(CHECKPOINT, map_location=DEVICE)
    dnn_to_bnn(net, BNN_PARAMS)
    net.load_state_dict(ckpt["model_state_dict"])
    mp = MomentPropEfficientnet_b0(net, num_mc=20)
    mp.to(DEVICE).eval()
    return mp

# Single forward pass complet
def single_pass(net, loader, is_mp=False):
    all_probs, all_labels = [], []
    with torch.no_grad():
        for inputs, labels in loader:
            inputs = inputs.to(DEVICE)
            labels = labels.squeeze(1)
            out = net(inputs)
            if is_mp:
                probs = F.softmax(out, dim=-1).mean(dim=0)   
            else:
                probs = F.softmax(out, dim=-1)                
            all_probs.append(probs.cpu().numpy())
            all_labels.append(labels.numpy())
    return np.concatenate(all_probs, axis=0), np.concatenate(all_labels, axis=0)

# N runs crash-safe
def collect_runs(net, loader, n_repeats, tag, is_mp=False):
    run_dir = OUTPUT_DIR / tag
    run_dir.mkdir(exist_ok=True)
    labels_path = run_dir / "labels.npy"

    outputs = []
    for i in range(n_repeats):
        run_path = run_dir / f"run_{i:03d}.npy"
        if run_path.exists():
            probs = np.load(run_path)
            if i == 0: print(f"  [{tag}] run {i+1} (cache)")
        else:
            t0 = time.time()
            probs, lbl = single_pass(net, loader, is_mp=is_mp)
            np.save(str(run_path) + ".tmp.npy", probs)
            shutil.move(str(run_path) + ".tmp.npy", run_path)
            if not labels_path.exists():
                np.save(labels_path, lbl)
            print(f"  [{tag}] run {i+1}/{n_repeats} — {time.time()-t0:.1f}s", flush=True)
        outputs.append(probs)

    labels = np.load(labels_path)
    return np.stack(outputs, axis=0), labels   # (N, dataset_size, C)

# MI / entropie (BALD)
def predictive_entropy(output):
    # output : (N, dataset_size, C)
    mean_p = output.mean(axis=0)
    return -np.sum(mean_p * np.log(mean_p + 1e-10), axis=-1)   

def mutual_information(output):
    mean_p = output.mean(axis=0)
    H      = -np.sum(mean_p * np.log(mean_p + 1e-10), axis=-1)
    exp_H  = -np.mean(np.sum(output * np.log(output + 1e-10), axis=-1), axis=0)
    return H - exp_H 

# MI : mean ± std correct/wrong × n_reps
def compute_mi_table(all_runs, all_labels, repeats_list):
    results = {}
    for n in repeats_list:
        sub    = all_runs[:n]                     
        mi     = mutual_information(sub)         
        preds  = sub.mean(axis=0).argmax(axis=-1) 
        correct = (preds == all_labels)

        def ms(mask):
            v = mi[mask]
            if len(v) == 0: return float("nan"), float("nan")
            return float(v.mean()), float(v.std())

        results[n] = {"correct": ms(correct), "wrong": ms(~correct)}
    return results

# Bar chart : H + MI per class
def make_bar_chart(all_runs, all_labels, title):
    mi = mutual_information(all_runs)
    H  = predictive_entropy(all_runs)

    h_cls, mi_cls = [], []
    for c in range(NUM_CLASSES):
        mask = (all_labels == c)
        h_cls.append( float(H[mask].mean())  if mask.any() else 0.0)
        mi_cls.append(float(mi[mask].mean()) if mask.any() else 0.0)

    x = np.arange(NUM_CLASSES)
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.bar(x, h_cls,   label="H — Predictive Entropy", color="#4472C4")
    ax.bar(x, mi_cls,  bottom=h_cls, label="MI — BALD (epistemic)", color="#ED7D31")
    ax.set_xticks(x)
    ax.set_xticklabels([f"Class {i}" for i in range(NUM_CLASSES)])
    ax.set_ylabel("Nats")
    ax.set_title(title)
    ax.legend(); ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=130, bbox_inches="tight")
    plt.close(fig); buf.seek(0)
    return buf

# Excel
thin   = Side(style="thin", color="CCCCCC")
border = Border(left=thin, right=thin, top=thin, bottom=thin)
hfill  = PatternFill("solid", start_color="2F4F8F")
cfill  = PatternFill("solid", start_color="D9EAD3")
wfill  = PatternFill("solid", start_color="F4CCCC")
hf     = Font(bold=True, color="FFFFFF", name="Arial", size=11)
nf     = Font(name="Arial", size=10)
nfb    = Font(name="Arial", size=10, bold=True)

def sh(cell, v, fill=None):
    cell.value = v; cell.font = hf
    cell.fill  = fill or hfill
    cell.alignment = Alignment(horizontal="center", vertical="center")
    cell.border = border

def nc(cell, v, bold=False, fill=None):
    cell.value = v; cell.font = nfb if bold else nf
    cell.alignment = Alignment(horizontal="center")
    cell.border = border
    if fill: cell.fill = fill

def write_mi_block(ws, row, model_label, split_name, results, repeats_list):
    ws.merge_cells(start_row=row, start_column=1, end_row=row, end_column=7)
    c = ws.cell(row, 1, f"{model_label} — {split_name}")
    c.font = Font(bold=True, size=12, name="Arial")
    c.alignment = Alignment(horizontal="left")
    row += 1

    sh(ws.cell(row, 1), "")
    for col_i, n in enumerate(repeats_list):
        base = 2 + col_i * 2
        ws.merge_cells(start_row=row, start_column=base, end_row=row, end_column=base+1)
        sh(ws.cell(row, base), f"n_reps = {n}")
    row += 1

    sh(ws.cell(row, 1), "")
    for col_i in range(len(repeats_list)):
        base = 2 + col_i * 2
        sh(ws.cell(row, base), "Mean MI"); sh(ws.cell(row, base+1), "Std MI")
    row += 1

    for case, label, fill in [("correct", "✓ Correct", cfill), ("wrong", "✗ Wrong", wfill)]:
        nc(ws.cell(row, 1), label, bold=True, fill=fill)
        for col_i, n in enumerate(repeats_list):
            mean_v, std_v = results[n][case]
            base = 2 + col_i * 2
            nc(ws.cell(row, base),   f"{mean_v:.4f}" if not np.isnan(mean_v) else "N/A")
            nc(ws.cell(row, base+1), f"{std_v:.4f}"  if not np.isnan(std_v)  else "N/A")
        row += 1
    return row + 2

# Main
print("Chargement des modèles...")
mc_model = load_mc_model()
mp_model = load_mp_model()

wb = openpyxl.Workbook()
wb.active.title = "Summary"
chart_ws  = wb.create_sheet("Charts")
chart_row = 1

MAX_REPS = max(REPEATS_LIST)

for split in ["train", "val", "test"]:
    print(f"\n{'='*55}\nSplit : {split}\n{'='*55}")
    loader = loaders[split]

    print("  → MC model...")
    runs_mc, labels = collect_runs(mc_model, loader, MAX_REPS, f"mc_{split}", is_mp=False)
    print("  → MomentProp model...")
    runs_mp, _      = collect_runs(mp_model, loader, MAX_REPS, f"mp_{split}", is_mp=True)

    res_mc = compute_mi_table(runs_mc, labels, REPEATS_LIST)
    res_mp = compute_mi_table(runs_mp, labels, REPEATS_LIST)

    ws = wb.create_sheet(f"MI_{split}")
    ws.column_dimensions["A"].width = 16
    for col in range(2, 9):
        ws.column_dimensions[get_column_letter(col)].width = 18

    row = 1
    row = write_mi_block(ws, row, "BayNoMoped (MC)", split, res_mc, REPEATS_LIST)
    row = write_mi_block(ws, row, "MomentProp",      split, res_mp, REPEATS_LIST)

    for model_label, runs in [("BayNoMoped_MC", runs_mc), ("MomentProp", runs_mp)]:
        buf = make_bar_chart(runs, labels, f"{model_label} — {split} (n={MAX_REPS})")
        img = XLImage(buf); img.anchor = f"A{chart_row}"
        chart_ws.add_image(img)
        chart_row += 30

wb.save("mi_analysis.xlsx")
print("\n✓ Excel généré : mi_analysis.xlsx")

Device : cuda
Chargement des modèles...

Split : train
  → MC model...
  [mc_train] run 1/100 — 11.1s
  [mc_train] run 2/100 — 11.3s
  [mc_train] run 3/100 — 11.5s
  [mc_train] run 4/100 — 11.0s
  [mc_train] run 5/100 — 11.2s
  [mc_train] run 6/100 — 10.7s
  [mc_train] run 7/100 — 11.2s
  [mc_train] run 8/100 — 11.1s
  [mc_train] run 9/100 — 11.7s
  [mc_train] run 10/100 — 11.4s
  [mc_train] run 11/100 — 11.0s
  [mc_train] run 12/100 — 11.0s
  [mc_train] run 13/100 — 11.3s
  [mc_train] run 14/100 — 11.1s
  [mc_train] run 15/100 — 11.1s
  [mc_train] run 16/100 — 11.0s
  [mc_train] run 17/100 — 11.0s
  [mc_train] run 18/100 — 11.1s
  [mc_train] run 19/100 — 11.9s
  [mc_train] run 20/100 — 10.6s
  [mc_train] run 21/100 — 10.9s
  [mc_train] run 22/100 — 11.2s
  [mc_train] run 23/100 — 11.2s
  [mc_train] run 24/100 — 10.7s
  [mc_train] run 25/100 — 9.9s
  [mc_train] run 26/100 — 10.2s
  [mc_train] run 27/100 — 10.0s
  [mc_train] run 28/100 — 9.8s
  [mc_train] run 29/100 — 10.2s
  [mc_train]